#  EDA — Browser-Use Evaluation Results
**File:** `results_20260626_042946.csv`  
**Model:** `google/gemma-4-E4B-it`  
**Analysis:** Descriptive + Diagnostic 

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# ── Absolute path — works regardless of notebook working directory ──
DATA_PATH = Path('results_20260626_042946.json')
import json
with open(DATA_PATH, 'r') as f:
    df = pd.DataFrame(json.load(f)['tasks'])

# ── Feature engineering ──
bool_cols = ['has_captcha_signal', 'has_auth_signal', 'has_timeout_signal', 'has_bot_block_signal']
for c in bool_cols:
    df[c] = df[c].astype(bool)

df['run_timestamp'] = pd.to_datetime(df['run_timestamp'], utc=True)

print(f' Loaded {len(df)} rows × {len(df.columns)} columns')
print(f'   Data path: {DATA_PATH}')
df.dtypes

In [ ]:
# ════════════════════════════════════════════════════════
# DATA QUALITY FIX — Correcting Hallucinated SUCCESS labels
# Agent admits failure in final_answer but status = SUCCESS
# ════════════════════════════════════════════════════════
import re

FAILURE_PHRASES = [
    r'i was unable to',
    r'unable to complete',
    r'could not complete',
    r'was not completed',
    r'could not be completed',
    r'could not achieve',
    r'not fully completed',
    r'task is incomplete',
    r'task could not',
    r'not completed',
    r'unable to log',
    r'unable to find',
    r'i could not',
    r'could not successfully',
    r'could not access',
    r'failed to complete',
    r'not able to',
    r'could not log',
    r'authentication.*failed',
    r'task is blocked',
]

def is_hallucinated_success(row):
    """Return True if a SUCCESS task's final_answer admits failure."""
    if row['status'] != 'SUCCESS':
        return False
    answer = str(row['final_answer'] or '').lower() + ' ' + str(row.get('agent_thoughts', '') or '').lower()
    return any(re.search(p, answer) for p in FAILURE_PHRASES)

df['is_hallucinated'] = df.apply(is_hallucinated_success, axis=1)
df['corrected_status'] = df.apply(
    lambda r: 'HALLUCINATED_SUCCESS' if r['is_hallucinated'] else r['status'], axis=1
)
df['is_success'] = df['corrected_status'] == 'SUCCESS'  # ← overwrite with corrected flag

# ── Summary ──
total          = len(df)
raw_success    = (df['status'] == 'SUCCESS').sum()
hallucinated   = df['is_hallucinated'].sum()
true_success   = df['is_success'].sum()

print('━'*55)
print('  DATA QUALITY CORRECTION APPLIED')
print('━'*55)
print(f'  Total tasks:              {total}')
print(f'  Raw SUCCESS labels:       {raw_success}  ({raw_success/total*100:.1f}%)')
print(f'  Hallucinated SUCCESS:     {hallucinated}  ({hallucinated/raw_success*100:.1f}% of labeled successes)')
print(f'  Genuine SUCCESS:       {true_success}  ({true_success/total*100:.1f}%)')
print(f'  FAILED:                {(df["corrected_status"]=="FAILED").sum()}')
print(f'  HALLUCINATED_SUCCESS: {hallucinated}')
print('━'*55)
print(f'\nAll charts below use corrected_status — hallucinated successes are treated as FAILED.')

df[['task_id','url_domain','category','corrected_status','is_hallucinated']].head(5)

## Data Quality — Hallucinated SUCCESS Labels
Agent marks tasks SUCCESS but final answer admits failure. These are corrected below.

In [ ]:
# ══ Hallucinated SUCCESS — Detail View ══
hallucinated_df = df[df['is_hallucinated']].copy()

print(f'Hallucinated SUCCESS tasks: {len(hallucinated_df)}')
print()

# ── Chart 1: Hallucinated vs Genuine by category ──
status_cat = df.groupby(['category','corrected_status']).size().reset_index(name='count')
color_map = {
    'SUCCESS':              '#10B981',
    'FAILED':               '#EF4444',
    'HALLUCINATED_SUCCESS': '#F59E0B',
}
fig1 = px.bar(
    status_cat,
    x='category', y='count', color='corrected_status',
    color_discrete_map=color_map,
    barmode='stack',
    text='count',
    title='Corrected Status Breakdown by Category<br><sup> HALLUCINATED_SUCCESS = agent admitted failure but was labelled SUCCESS</sup>',
    labels={'count': 'Tasks', 'corrected_status': 'Status'}
)
fig1.update_traces(textposition='inside', textfont_size=13, textfont_color='white',
                   insidetextanchor='middle')
fig1.update_layout(
    height=420, plot_bgcolor='#f8fafc', paper_bgcolor='white',
    legend=dict(title='Status', font=dict(size=13), orientation='h',
                yanchor='bottom', y=1.02, xanchor='right', x=1),
    xaxis=dict(tickfont=dict(size=13, color='#1e293b')),
    yaxis=dict(title='Number of Tasks', tickfont=dict(size=12),
               showgrid=True, gridcolor='rgba(0,0,0,0.07)'),
    title_font=dict(size=15, color='#1e293b'),
    title_x=0.5
)
fig1.show()

# ── Chart 2: Raw vs Corrected success rate ──
raw_sr    = (df['status'] == 'SUCCESS').mean() * 100
true_sr   = df['is_success'].mean() * 100
halluc_sr = df['is_hallucinated'].mean() * 100

fig2 = go.Figure()
fig2.add_trace(go.Bar(
    x=['Raw Labeled SR', 'Genuine SR (corrected)', 'Hallucinated portion'],
    y=[raw_sr, true_sr, halluc_sr],
    marker_color=['#3B82F6', '#10B981', '#F59E0B'],
    text=[f'{v:.1f}%' for v in [raw_sr, true_sr, halluc_sr]],
    textposition='outside',
    textfont=dict(size=14, color='#1e293b'),
    cliponaxis=False,
))
fig2.update_layout(
    title=dict(text='Raw vs Corrected Success Rate',
               font=dict(size=15, color='#1e293b'), x=0.5, xanchor='center'),
    yaxis=dict(title='Rate (%)', range=[0, 75],
               showgrid=True, gridcolor='rgba(0,0,0,0.07)', tickfont=dict(size=12)),
    xaxis=dict(tickfont=dict(size=13, color='#1e293b')),
    height=380, plot_bgcolor='#f8fafc', paper_bgcolor='white',
    showlegend=False, margin=dict(t=70, b=50, l=50, r=50)
)
fig2.show()

# ── Table: all hallucinated tasks ──
cols = ['task_id','category','url_domain','steps_taken','final_answer']
print('\nAll hallucinated SUCCESS tasks:')
for _, r in hallucinated_df[cols].iterrows():
    snippet = str(r['final_answer'])[:120].replace('\n',' ')
    print(f"  [{r['task_id']}] {r['category']} | {r['url_domain']}")
    print(f"    → {snippet!r}")

Hallucinated SUCCESS tasks: 26




All hallucinated SUCCESS tasks:
  [task_0007] READ | my.clevelandclinic.org
    → 'I attempted to fulfill your request by navigating to https://my.clevelandclinic.org and exploring resources related to m'
  [task_0013] READ | www.yale.edu
    → "I was unable to locate a specific 'Research Highlights' page on yale.edu or its related domains using only navigation wi"
  [task_0019] READ | www.who.int
    → 'I was unable to locate the latest global health report on COVID-19 and list its main findings using only http://who.int '
  [task_0031] READ | www.wikihow.com
    → 'I was unable to complete the request because wikiHow.com does not provide an easily accessible search results page when '
  [task_0033] READ | www.apple.com
    → 'I was unable to definitively find out if unlimited repairs are included with the AirPods Pro using only apple.com. After'
  [task_0038] READ | www.istockphoto.com
    → "The task required navigating to iStockphoto.com, searching for 'waterfalls' videos, applyin

##  SECTION 1 — DESCRIPTIVE ANALYSIS
### 1A. Dataset Overview

In [ ]:
print('=== DATASET OVERVIEW ===')
print(f'Shape: {df.shape}')
print(f'\nNull counts (>0 only):')
nulls = df.isnull().sum()
print(nulls[nulls > 0].to_string())

print(f'\nUnique values per categorical column:')
cat_cols = df.select_dtypes('object').columns
for c in cat_cols:
    print(f'  {c}: {df[c].nunique()} unique values')

=== DATASET OVERVIEW ===
Shape: (113, 38)

Null counts (>0 only):
final_answer             16
error_code               50
error_label              50
error_hint               50
error_message            97
failure_root             50
task_validity_reason     50
url_probe_note          102

Unique values per categorical column:
  task_id: 113 unique values
  category: 5 unique values
  url: 34 unique values
  url_domain: 34 unique values
  instruction: 113 unique values
  status: 2 unique values
  llm_model: 1 unique values
  llm_provider: 1 unique values
  final_answer: 97 unique values
  error_code: 6 unique values
  error_label: 6 unique values
  error_hint: 6 unique values
  error_message: 5 unique values
  failure_root: 3 unique values
  task_validity_code: 6 unique values
  task_validity_reason: 6 unique values
  url_probe_note: 2 unique values
  corrected_status: 3 unique values


### 1B. Status & Category Distribution

In [ ]:
# ── Status distribution ──
status_counts = df['corrected_status'].value_counts().reset_index()
status_counts.columns = ['Status', 'Count']
status_counts['Pct'] = (status_counts['Count'] / len(df) * 100).round(1)

# Chart 1 — Status donut (standalone, no label collision)
fig_status = go.Figure(go.Pie(
    labels=status_counts['Status'],
    values=status_counts['Count'],
    marker=dict(colors=[{'SUCCESS': '#10B981', 'FAILED': '#EF4444', 'HALLUCINATED_SUCCESS': '#F59E0B'}[s] for s in status_counts['Status']]),
    textinfo='label+percent+value',
    textfont=dict(size=15, color='white'),
    textposition='inside',
    insidetextorientation='radial',
    hole=0.42,
    pull=[0.04, 0],
))
fig_status.update_layout(
    title=dict(text='Overall Task Outcome — 113 Tasks',
               font=dict(size=17, color='#1e293b'), x=0.5, xanchor='center'),
    height=380,
    paper_bgcolor='white',
    legend=dict(font=dict(size=14), orientation='h', yanchor='bottom', y=-0.12, xanchor='center', x=0.5),
    margin=dict(l=40, r=40, t=70, b=60)
)
fig_status.show()

# Chart 2 — Category success rate (horizontal bar — avoids label overlap)
cat_sr = df.groupby('category')['is_success'].agg(['sum','count','mean']).reset_index()
cat_sr.columns = ['category','successes','total','success_rate']
cat_sr['success_pct'] = (cat_sr['success_rate'] * 100).round(1)
cat_sr = cat_sr.sort_values('success_pct')

bar_colors = ['#EF4444' if x < 40 else '#F59E0B' if x < 55 else '#10B981'
              for x in cat_sr['success_pct']]

fig_cat = go.Figure(go.Bar(
    x=cat_sr['success_pct'],
    y=cat_sr['category'],
    orientation='h',
    marker=dict(color=bar_colors, line=dict(color='rgba(255,255,255,0.4)', width=1)),
    text=[f'  {p}%  ({s}/{t} tasks)'
          for p, s, t in zip(cat_sr['success_pct'], cat_sr['successes'], cat_sr['total'])],
    textposition='outside',
    textfont=dict(size=13, color='#1e293b'),
    cliponaxis=False,
))
fig_cat.add_vline(x=44.2, line_dash='dash', line_color='#94a3b8',
                  annotation_text='Overall 44.2%',
                  annotation_font=dict(size=12, color='#64748b'))
fig_cat.update_layout(
    title=dict(text='Success Rate by Task Category',
               font=dict(size=17, color='#1e293b'), x=0.5, xanchor='center'),
    xaxis=dict(
        title='Success Rate (%)', range=[0, 110],
        tickfont=dict(size=12), showgrid=True, gridcolor='rgba(0,0,0,0.07)'
    ),
    yaxis=dict(tickfont=dict(size=14, color='#1e293b'), automargin=True),
    height=340,
    plot_bgcolor='#f8fafc',
    paper_bgcolor='white',
    margin=dict(l=20, r=120, t=70, b=50),
    showlegend=False
)
fig_cat.show()

print('\nStatus breakdown:')
print(status_counts.to_string(index=False))
print('\nCategory success rates:')
print(cat_sr[['category','total','successes','success_pct']].sort_values('success_pct', ascending=False).to_string(index=False))


Status breakdown:
              Status  Count  Pct
              FAILED     63 55.8
HALLUCINATED_SUCCESS     26 23.0
             SUCCESS     24 21.2

Category success rates:
         category  total  successes  success_pct
             READ     82         22         26.8
FILE_MANIPULATION      5          1         20.0
           UPDATE      8          1         12.5
           CREATE     12          0          0.0
           DELETE      6          0          0.0


### 1C. Quantitative Variable Distributions

In [ ]:
# ── Descriptive stats table ──
quant_cols = [
    'latency_seconds', 'steps_taken', 'steps_utilization_pct',
    'retry_count', 'llm_calls', 'total_thought_chars',
    'final_answer_length', 'avg_step_latency_seconds',
    'instruction_length', 'task_validity_confidence'
]
desc = df[quant_cols].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]).T
desc.columns = ['n', 'mean', 'std', 'min', 'p25', 'p50', 'p75', 'p90', 'p95', 'max']
print('Descriptive Statistics:')
print(desc.round(2).to_string())

Descriptive Statistics:
                              n      mean       std    min       p25       p50       p75       p90       p95       max
latency_seconds           113.0    972.70    689.03    0.0    368.75    838.07   1602.22   1884.40   2067.57   2402.32
steps_taken               113.0     13.01      7.09    0.0      7.00     15.00     20.00     20.00     20.00     21.00
steps_utilization_pct     113.0     65.04     35.45    0.0     35.00     75.00    100.00    100.00    100.00    105.00
retry_count               113.0      0.37      0.73    0.0      0.00      0.00      0.00      2.00      2.00      2.00
llm_calls                 113.0     11.93      6.92    0.0      6.00     13.00     19.00     20.00     20.00     20.00
total_thought_chars       113.0  24024.86  15124.93    0.0  10590.00  27325.00  37334.00  41842.20  43749.20  52666.00
final_answer_length       113.0    461.62    313.25    0.0    298.00    414.00    605.00    854.40   1082.20   1478.00
avg_step_latency_seconds

In [ ]:
# ── Latency distribution ──
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Latency Distribution (all tasks)', 'Latency by Status (box)'])

for status, color in [('SUCCESS', '#10B981'), ('FAILED', '#EF4444'), ('HALLUCINATED_SUCCESS', '#F59E0B')]:
    subset = df[df['corrected_status'] == status]
    fig.add_trace(go.Histogram(
        x=subset['latency_seconds'],
        name=status,
        marker_color=color,
        opacity=0.7,
        nbinsx=25
    ), row=1, col=1)

for status, color in [('SUCCESS', '#10B981'), ('FAILED', '#EF4444'), ('HALLUCINATED_SUCCESS', '#F59E0B')]:
    subset = df[df['corrected_status'] == status]
    fig.add_trace(go.Box(
        y=subset['latency_seconds'],
        name=status,
        marker_color=color,
        boxmean=True
    ), row=1, col=2)

fig.update_layout(
    title_text='Latency Analysis',
    height=400,
    barmode='overlay',
    legend_title='Status'
)
fig.update_xaxes(title_text='Latency (seconds)', row=1, col=1)
fig.update_yaxes(title_text='Count', row=1, col=1)
fig.update_yaxes(title_text='Latency (seconds)', row=1, col=2)
fig.show()

print('\nLatency breakdown by status:')
lat_by_status = df.groupby('corrected_status')['latency_seconds'].agg(['mean', 'median', 'std', 'min', 'max'])
print(lat_by_status.round(1).to_string())


Latency breakdown by status:
                        mean  median    std    min     max
corrected_status                                          
FAILED                 983.9   840.4  749.5    0.0  2402.3
HALLUCINATED_SUCCESS  1211.2  1217.8  596.8  129.2  2257.8
SUCCESS                684.8   529.9  509.2  112.5  2258.6


In [ ]:
# ── Steps & LLM calls distribution ──
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Steps Taken Distribution', 'LLM Calls Distribution'])

for status, color in [('SUCCESS', '#10B981'), ('FAILED', '#EF4444'), ('HALLUCINATED_SUCCESS', '#F59E0B')]:
    subset = df[df['corrected_status'] == status]
    fig.add_trace(go.Histogram(
        x=subset['steps_taken'], name=status,
        marker_color=color, opacity=0.7, nbinsx=20
    ), row=1, col=1)
    fig.add_trace(go.Histogram(
        x=subset['llm_calls'], name=status,
        marker_color=color, opacity=0.7, nbinsx=20,
        showlegend=False
    ), row=1, col=2)

fig.update_layout(title_text='Agent Activity Distribution', height=400, barmode='overlay')
fig.update_xaxes(title_text='Steps Taken', row=1, col=1)
fig.update_xaxes(title_text='LLM Calls', row=1, col=2)
fig.show()

### 1D. Domain & Retry Analysis

In [ ]:
# ── Domain success rates ──
domain_perf = df.groupby('url_domain').agg(
    total=('is_success', 'count'),
    successes=('is_success', 'sum'),
    success_rate=('is_success', 'mean'),
    avg_latency=('latency_seconds', 'mean')
).reset_index()
domain_perf = domain_perf[domain_perf['total'] >= 3].sort_values('success_rate')
domain_perf['success_pct'] = (domain_perf['success_rate'] * 100).round(1)
# Strip 'www.' for cleaner y-axis labels
domain_perf['label'] = domain_perf['url_domain'].str.replace('www.', '', regex=False)

bar_colors = ['#EF4444' if x <= 20 else '#F59E0B' if x <= 50 else '#10B981'
              for x in domain_perf['success_pct']]

fig = go.Figure(go.Bar(
    x=domain_perf['success_pct'],
    y=domain_perf['label'],
    orientation='h',
    marker=dict(color=bar_colors, line=dict(color='rgba(255,255,255,0.4)', width=1)),
    # Put text INSIDE the bar so it never overlaps the axis
    text=[f'{p}%  ({s}/{t})'
          for p, s, t in zip(domain_perf['success_pct'],
                             domain_perf['successes'],
                             domain_perf['total'])],
    textposition='auto',
    textfont=dict(size=12, color='white'),
    cliponaxis=False,
))
fig.add_vline(x=44.2, line_dash='dash', line_color='#64748b',
              annotation_text='Avg 44.2%',
              annotation_font=dict(size=11, color='#64748b'),
              annotation_position='top right')
fig.update_layout(
    title=dict(text='Success Rate by Domain  (≥ 3 tasks)  |  Red < 20%  ·  Yellow ≤ 50%  ·  Green > 50%',
               font=dict(size=15, color='#1e293b'), x=0.5, xanchor='center'),
    xaxis=dict(
        title='Success Rate (%)', range=[0, 100],
        tickfont=dict(size=12), showgrid=True, gridcolor='rgba(0,0,0,0.07)',
        dtick=20
    ),
    yaxis=dict(tickfont=dict(size=12, color='#1e293b'), automargin=True),
    height=520,
    plot_bgcolor='#f8fafc',
    paper_bgcolor='white',
    margin=dict(l=20, r=30, t=80, b=50),
    showlegend=False
)
fig.show()

In [ ]:
# ── Retry impact ──
retry_impact = df.groupby('retry_count').agg(
    total=('is_success', 'count'),
    successes=('is_success', 'sum'),
    success_rate=('is_success', 'mean'),
    avg_latency=('latency_seconds', 'mean')
).reset_index()
retry_impact['success_pct'] = (retry_impact['success_rate'] * 100).round(1)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Success Rate by Retry Count', 'Avg Latency by Retry Count'])

fig.add_trace(go.Bar(
    x=retry_impact['retry_count'],
    y=retry_impact['success_pct'],
    marker_color=['#10B981', '#F59E0B', '#EF4444'],
    text=[f'{v}%<br>(n={t})' for v, t in zip(retry_impact['success_pct'], retry_impact['total'])],
    textposition='outside'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=retry_impact['retry_count'],
    y=retry_impact['avg_latency'].round(0),
    marker_color=['#3B82F6', '#8B5CF6', '#EC4899'],
    text=[f'{int(v)}s' for v in retry_impact['avg_latency']],
    textposition='outside'
), row=1, col=2)

fig.update_layout(title_text='Retry Impact Analysis', height=400, showlegend=False)
fig.update_xaxes(title_text='Retry Count', tickvals=[0, 1, 2])
fig.update_yaxes(title_text='Success Rate (%)', range=[0, 90], row=1, col=1)
fig.update_yaxes(title_text='Avg Latency (s)', row=1, col=2)
fig.show()

print('Retry breakdown:')
print(retry_impact[['retry_count', 'total', 'successes', 'success_pct', 'avg_latency']].to_string(index=False))

Retry breakdown:
 retry_count  total  successes  success_pct  avg_latency
           0     88         19         21.6   994.629659
           1      8          2         25.0  1639.581250
           2     17          3         17.6   545.358235


In [ ]:
# ── Retry impact ──
retry_impact = df.groupby('retry_count').agg(
    total=('is_success', 'count'),
    successes=('is_success', 'sum'),
    success_rate=('is_success', 'mean'),
    avg_latency=('latency_seconds', 'mean')
).reset_index()
retry_impact['success_pct'] = (retry_impact['success_rate'] * 100).round(1)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Success Rate by Retry Count', 'Avg Latency by Retry Count'])

fig.add_trace(go.Bar(
    x=retry_impact['retry_count'],
    y=retry_impact['success_pct'],
    marker_color=['#10B981', '#F59E0B', '#EF4444'],
    text=[f'{v}%<br>(n={t})' for v, t in zip(retry_impact['success_pct'], retry_impact['total'])],
    textposition='outside'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=retry_impact['retry_count'],
    y=retry_impact['avg_latency'].round(0),
    marker_color=['#3B82F6', '#8B5CF6', '#EC4899'],
    text=[f'{int(v)}s' for v in retry_impact['avg_latency']],
    textposition='outside'
), row=1, col=2)

fig.update_layout(title_text='Retry Impact Analysis', height=400, showlegend=False)
fig.update_xaxes(title_text='Retry Count', tickvals=[0, 1, 2])
fig.update_yaxes(title_text='Success Rate (%)', range=[0, 90], row=1, col=1)
fig.update_yaxes(title_text='Avg Latency (s)', row=1, col=2)
fig.show()

print('Retry breakdown:')
print(retry_impact[['retry_count', 'total', 'successes', 'success_pct', 'avg_latency']].to_string(index=False))

Retry breakdown:
 retry_count  total  successes  success_pct  avg_latency
           0     88         19         21.6   994.629659
           1      8          2         25.0  1639.581250
           2     17          3         17.6   545.358235


##  SECTION 2 — DIAGNOSTIC ANALYSIS
### 2A. Root Cause Taxonomy

In [ ]:
failed = df[df['corrected_status'] == 'FAILED'].copy()
print(f'Failed tasks: {len(failed)} / {len(df)} ({len(failed)/len(df)*100:.1f}%)')

# ── Short label helper ──
label_map = {
    'Data_StaleLayout':              'DOM Stale Layout',
    'Logic_AgentLoopOrHallucination':'Agent Loop / Hallucination',
    'System_SilentCrash':            'Silent Crash',
    'Auth_CredentialInjectionFailed':'Auth / No Credentials',
    'Security_BotBlocked':           'Bot Blocked (WAF)',
    'Security_CaptchaFailed':        'CAPTCHA Failed',
    'FRAMEWORK_LIMIT':               'Framework Limit',
    'TASK_INVALID':                  'Task Invalid',
    'AGENT_FAILURE':                 'Agent Failure',
}

# ── Error codes (failed tasks) ──
ec = (failed['error_code']
      .value_counts(dropna=True)
      .rename(index=label_map)
      .reset_index())
ec.columns = ['Error Code', 'Count']
ec['Pct'] = (ec['Count'] / len(failed) * 100).round(1)
ec = ec.sort_values('Count')  # ascending so longest bar is at top

# ── Failure root (all tasks) ──
root = (df['failure_root']
        .value_counts(dropna=True)
        .rename(index=label_map)
        .reset_index())
root.columns = ['Root Cause', 'Count']
root['Pct'] = (root['Count'] / len(df) * 100).round(1)

# ═══════════════════════════════════════════
# Chart 1 — Error Code (horizontal bar)
# ═══════════════════════════════════════════
bar_colors = [
    '#EF4444', '#F97316', '#EAB308',
    '#8B5CF6', '#EC4899', '#14B8A6'
]

fig1 = go.Figure(go.Bar(
    x=ec['Count'],
    y=ec['Error Code'],
    orientation='h',
    marker=dict(
        color=bar_colors[:len(ec)],
        line=dict(color='rgba(255,255,255,0.3)', width=1)
    ),
    text=[f'  {c}  ({p}%)' for c, p in zip(ec['Count'], ec['Pct'])],
    textposition='outside',
    textfont=dict(size=13, color='#1e293b'),
    cliponaxis=False,
))
fig1.update_layout(
    title=dict(text='Error Code Breakdown — Failed Tasks (n=63)',
               font=dict(size=16, color='#1e293b'), x=0.5, xanchor='center'),
    xaxis=dict(
        title='Number of Tasks',
        range=[0, ec['Count'].max() * 1.35],
        tickfont=dict(size=12),
        showgrid=True, gridcolor='rgba(0,0,0,0.08)'
    ),
    yaxis=dict(
        tickfont=dict(size=13, color='#1e293b'),
        automargin=True
    ),
    height=380,
    plot_bgcolor='#f8fafc',
    paper_bgcolor='white',
    margin=dict(l=20, r=80, t=60, b=40),
    showlegend=False
)
fig1.show()

# ═══════════════════════════════════════════
# Chart 2 — Root Cause (horizontal bar)
# ═══════════════════════════════════════════
root_colors = ['#3B82F6', '#EF4444', '#F59E0B']

root_sorted = root.sort_values('Count')
fig2 = go.Figure(go.Bar(
    x=root_sorted['Count'],
    y=root_sorted['Root Cause'],
    orientation='h',
    marker=dict(
        color=root_colors[:len(root_sorted)],
        line=dict(color='rgba(255,255,255,0.3)', width=1)
    ),
    text=[f'  {c} tasks  ({p}%)' for c, p in zip(root_sorted['Count'], root_sorted['Pct'])],
    textposition='outside',
    textfont=dict(size=13, color='#1e293b'),
    cliponaxis=False,
))
fig2.update_layout(
    title=dict(text='Failure Root Cause — All Tasks (n=113)',
               font=dict(size=16, color='#1e293b'), x=0.5, xanchor='center'),
    xaxis=dict(
        title='Number of Tasks',
        range=[0, root_sorted['Count'].max() * 1.45],
        tickfont=dict(size=12),
        showgrid=True, gridcolor='rgba(0,0,0,0.08)'
    ),
    yaxis=dict(
        tickfont=dict(size=13, color='#1e293b'),
        automargin=True
    ),
    height=280,
    plot_bgcolor='#f8fafc',
    paper_bgcolor='white',
    margin=dict(l=20, r=120, t=60, b=40),
    showlegend=False
)
fig2.show()

print('\nError codes (failed tasks):')
print(ec[['Error Code','Count','Pct']].sort_values('Count', ascending=False).to_string(index=False))
print('\nFailure roots (all tasks):')
print(root.to_string(index=False))

Failed tasks: 63 / 113 (55.8%)



Error codes (failed tasks):
                Error Code  Count  Pct
          DOM Stale Layout     19 30.2
Agent Loop / Hallucination     19 30.2
              Silent Crash      9 14.3
     Auth / No Credentials      6  9.5
         Bot Blocked (WAF)      5  7.9
            CAPTCHA Failed      5  7.9

Failure roots (all tasks):
     Root Cause  Count  Pct
Framework Limit     49 43.4
   Task Invalid     11  9.7
  Agent Failure      3  2.7


### 2B. Task Validity & Adjusted Success Rate

In [ ]:
# ── Task validity breakdown ──
validity = df['task_validity_code'].value_counts().reset_index()
validity.columns = ['Validity Code', 'Count']
validity['Pct'] = (validity['Count'] / len(df) * 100).round(1)

print('Task Validity Distribution:')
print(validity.to_string(index=False))

# Adjusted success rate
valid_tasks = df[df['task_validity_code'] == 'VALID']
print(f'\n--- ADJUSTED SUCCESS RATE ---')
print(f'Valid tasks: {len(valid_tasks)} / {len(df)}')
print(f'Raw success rate:      {df["is_success"].mean()*100:.1f}%')
print(f'Adjusted success rate: {valid_tasks["is_success"].mean()*100:.1f}%')

# ── Chart 1: Validity distribution (horizontal bar — no overlap) ──
validity_sorted = validity.sort_values('Count')
v_colors = ['#10B981','#3B82F6','#F59E0B','#EF4444','#8B5CF6','#EC4899']

fig_v = go.Figure(go.Bar(
    x=validity_sorted['Count'],
    y=validity_sorted['Validity Code'],
    orientation='h',
    marker=dict(color=v_colors[:len(validity_sorted)],
                line=dict(color='rgba(255,255,255,0.4)', width=1)),
    text=[f'  {c} tasks ({p}%)' for c, p in zip(validity_sorted['Count'], validity_sorted['Pct'])],
    textposition='outside',
    textfont=dict(size=13, color='#1e293b'),
    cliponaxis=False,
))
fig_v.update_layout(
    title=dict(text='Task Validity Code Distribution',
               font=dict(size=16, color='#1e293b'), x=0.5, xanchor='center'),
    xaxis=dict(
        title='Number of Tasks',
        range=[0, validity_sorted['Count'].max() * 1.45],
        tickfont=dict(size=12), showgrid=True, gridcolor='rgba(0,0,0,0.07)'
    ),
    yaxis=dict(tickfont=dict(size=13, color='#1e293b'), automargin=True),
    height=360,
    plot_bgcolor='#f8fafc',
    paper_bgcolor='white',
    margin=dict(l=20, r=120, t=70, b=50),
    showlegend=False
)
fig_v.show()

# ── Chart 2: Validity × Status 100%-stacked (with outside labels removed, use legend) ──
vx_status = pd.crosstab(df['task_validity_code'], df['corrected_status'])
vx_status_pct = vx_status.div(vx_status.sum(axis=1), axis=0) * 100

fig_vx = go.Figure()
colors_map = {'SUCCESS': '#10B981', 'FAILED': '#EF4444', 'HALLUCINATED_SUCCESS': '#F59E0B'}
for status in ['SUCCESS', 'FAILED', 'HALLUCINATED_SUCCESS']:
    if status in vx_status_pct.columns:
        vals = vx_status_pct[status].round(1)
        fig_vx.add_trace(go.Bar(
            name=status,
            x=vx_status_pct.index,
            y=vals,
            marker_color=colors_map[status],
            # Only show text where bar is wide enough (>=15%)
            text=[f'{v:.0f}%' if v >= 15 else '' for v in vals],
            textposition='inside',
            textfont=dict(size=13, color='white'),
            insidetextanchor='middle',
        ))
fig_vx.update_layout(
    title=dict(text='Success vs Failure Rate by Validity Code (100% stacked)',
               font=dict(size=16, color='#1e293b'), x=0.5, xanchor='center'),
    barmode='stack',
    yaxis=dict(title='Percentage (%)', range=[0, 105], tickfont=dict(size=12)),
    xaxis=dict(tickfont=dict(size=12, color='#1e293b'), automargin=True,
               tickangle=-20),
    legend=dict(font=dict(size=13), title='Status', orientation='h',
                yanchor='bottom', y=1.02, xanchor='right', x=1),
    height=380,
    plot_bgcolor='#f8fafc',
    paper_bgcolor='white',
    margin=dict(l=50, r=30, t=80, b=80),
)
fig_vx.show()

Task Validity Distribution:
            Validity Code  Count  Pct
                    VALID     50 44.2
 Framework_NoFileHandling     47 41.6
                 URL_Dead     11  9.7
                  UNKNOWN      3  2.7
  Framework_NoCredentials      1  0.9
Framework_NoCaptchaSolver      1  0.9

--- ADJUSTED SUCCESS RATE ---
Valid tasks: 50 / 113
Raw success rate:      21.2%
Adjusted success rate: 48.0%


### 2C. Obstacle Signal Impact

In [ ]:
# ── Obstacle signal analysis ──
signal_data = []
for col in bool_cols:
    with_flag = df[df[col] == True]['is_success']
    without_flag = df[df[col] == False]['is_success']
    signal_data.append({
        'signal': col.replace('has_', '').replace('_signal', '').title(),
        'n_with': len(with_flag),
        'n_without': len(without_flag),
        'sr_with': with_flag.mean() * 100 if len(with_flag) > 0 else 0,
        'sr_without': without_flag.mean() * 100,
    })

sig_df = pd.DataFrame(signal_data)
sig_df['delta'] = sig_df['sr_with'] - sig_df['sr_without']

fig = go.Figure()
fig.add_trace(go.Bar(
    name='With Signal',
    x=sig_df['signal'],
    y=sig_df['sr_with'].round(1),
    marker_color='#F59E0B',
    text=[f'{v:.1f}%<br>(n={n})' for v, n in zip(sig_df['sr_with'], sig_df['n_with'])],
    textposition='outside'
))
fig.add_trace(go.Bar(
    name='Without Signal',
    x=sig_df['signal'],
    y=sig_df['sr_without'].round(1),
    marker_color='#3B82F6',
    text=[f'{v:.1f}%<br>(n={n})' for v, n in zip(sig_df['sr_without'], sig_df['n_without'])],
    textposition='outside'
))

fig.update_layout(
    title='Obstacle Signal Impact on Success Rate',
    barmode='group',
    yaxis=dict(title='Success Rate (%)', range=[0, 90]),
    height=420,
    legend_title='Condition'
)
fig.show()

print('Signal Impact Summary:')
print(sig_df[['signal', 'n_with', 'sr_with', 'n_without', 'sr_without', 'delta']].round(1).to_string(index=False))

Signal Impact Summary:
   signal  n_with  sr_with  n_without  sr_without  delta
  Captcha       8      0.0        105        22.9  -22.9
     Auth      29     10.3         84        25.0  -14.7
  Timeout       0      0.0        113        21.2  -21.2
Bot_Block       7      0.0        106        22.6  -22.6


### 2D. Step Budget Exhaustion Analysis

In [ ]:
# ── Step budget exhaustion ──
df['budget_exhausted'] = df['steps_utilization_pct'] >= 100
exhausted = df[df['budget_exhausted']]
not_exhausted = df[~df['budget_exhausted']]

print(f'Budget exhausted (≥100%): {len(exhausted)} tasks ({len(exhausted)/len(df)*100:.1f}%)')
print(f'  → Success rate: {exhausted["is_success"].mean()*100:.1f}%')
print(f'Not exhausted: {len(not_exhausted)} tasks')
print(f'  → Success rate: {not_exhausted["is_success"].mean()*100:.1f}%')

# Scatter: steps_utilization_pct vs success
fig = px.strip(
    df,
    x='steps_utilization_pct',
    y='corrected_status',
    color='corrected_status',
    color_discrete_map={'SUCCESS': '#10B981', 'FAILED': '#EF4444', 'HALLUCINATED_SUCCESS': '#F59E0B'},
    title='Step Utilization % vs Task Status (strip plot)',
    labels={'steps_utilization_pct': 'Steps Used (%)', 'corrected_status': 'Outcome'}
)
fig.add_vline(x=100, line_dash='dash', line_color='white', annotation_text='100% budget')
fig.update_layout(height=350)
fig.show()

Budget exhausted (≥100%): 35 tasks (31.0%)
  → Success rate: 5.7%
Not exhausted: 78 tasks
  → Success rate: 28.2%


### 2E. Category × Error Code Cross-Analysis

In [ ]:
# ── Category × Error code heatmap ──
cross = pd.crosstab(df['category'], df['error_code'])

fig = px.imshow(
    cross,
    text_auto=True,
    color_continuous_scale='RdYlGn_r',
    title='Category × Error Code Heatmap (task count)',
    labels=dict(x='Error Code', y='Category', color='Count')
)
fig.update_layout(height=400, xaxis_tickangle=-30)
fig.show()

print('\nCross-tab:')
print(cross.to_string())


Cross-tab:
error_code         Auth_CredentialInjectionFailed  Data_StaleLayout  Logic_AgentLoopOrHallucination  Security_BotBlocked  Security_CaptchaFailed  System_SilentCrash
category                                                                                                                                                            
CREATE                                          3                 2                               0                    0                       2                   0
DELETE                                          2                 0                               1                    0                       1                   0
FILE_MANIPULATION                               0                 2                               1                    0                       0                   0
READ                                            0                15                              15                    5                       2                   

### 2F. Correlation & Feature Relationships

In [ ]:
# ── Correlation heatmap ──
corr_cols = [
    'llm_calls', 'steps_taken', 'latency_seconds',
    'total_thought_chars', 'final_answer_length',
    'steps_utilization_pct', 'retry_count', 'instruction_length'
]
corr = df[corr_cols].corr()

fig = px.imshow(
    corr,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1,
    title='Correlation Matrix — Quantitative Features',
    aspect='auto'
)
fig.update_layout(height=500)
fig.show()

In [ ]:
# ── Scatter: LLM calls vs latency coloured by status ──
fig = px.scatter(
    df,
    x='llm_calls',
    y='latency_seconds',
    color='corrected_status',
    symbol='category',
    hover_data=['task_id', 'url_domain', 'steps_taken', 'error_code'],
    color_discrete_map={'SUCCESS': '#10B981', 'FAILED': '#EF4444', 'HALLUCINATED_SUCCESS': '#F59E0B'},
    title='LLM Calls vs Latency by Status & Category',
    labels={'llm_calls': 'LLM Calls', 'latency_seconds': 'Latency (s)'}
)
fig.update_layout(height=480)
fig.show()

### 2G. Thought Depth vs Answer Quality

In [ ]:
# ── Thought chars vs answer length ──
success = df[df['is_success']]
fail_tasks = df[~df['is_success']]

print('=== THOUGHT DEPTH vs ANSWER QUALITY ===')
print(f"Avg thought chars  — SUCCESS: {success['total_thought_chars'].mean():,.0f}  |  FAILED: {fail_tasks['total_thought_chars'].mean():,.0f}")
print(f"Avg answer length  — SUCCESS: {success['final_answer_length'].mean():,.0f}  |  FAILED: {fail_tasks['final_answer_length'].mean():,.0f} chars")
print(f"Avg LLM calls      — SUCCESS: {success['llm_calls'].mean():,.1f}      |  FAILED: {fail_tasks['llm_calls'].mean():,.1f}")

fig = px.scatter(
    df[df['final_answer_length'] > 0],
    x='total_thought_chars',
    y='final_answer_length',
    color='corrected_status',
    size='llm_calls',
    hover_data=['task_id', 'url_domain', 'category'],
    color_discrete_map={'SUCCESS': '#10B981', 'FAILED': '#EF4444', 'HALLUCINATED_SUCCESS': '#F59E0B'},
    trendline='ols',
    title='Thought Chars vs Answer Length (bubble size = LLM Calls)',
    labels={'total_thought_chars': 'Total Thought Characters', 'final_answer_length': 'Final Answer Length'}
)
fig.update_layout(height=450)
fig.show()

=== THOUGHT DEPTH vs ANSWER QUALITY ===
Avg thought chars  — SUCCESS: 15,916  |  FAILED: 26,212
Avg answer length  — SUCCESS: 580  |  FAILED: 430 chars
Avg LLM calls      — SUCCESS: 8.2      |  FAILED: 12.9


### 2H. Domain-Level Failure Pattern

In [ ]:
# ── Domain failure analysis ──
def safe_mode(x):
    x = x.dropna()
    return x.mode()[0] if len(x) > 0 else 'N/A'

# Short error code labels
err_short = {
    'Data_StaleLayout':              'DOM Stale Layout',
    'Logic_AgentLoopOrHallucination':'Agent Loop',
    'System_SilentCrash':            'Silent Crash',
    'Auth_CredentialInjectionFailed':'Auth Fail',
    'Security_BotBlocked':           'Bot Blocked',
    'Security_CaptchaFailed':        'CAPTCHA',
}

domain_fail = df.groupby('url_domain').agg(
    total=('is_success', 'count'),
    fail_rate=('is_success', lambda x: (1 - x.mean()) * 100),
    common_error=('error_code', safe_mode),
    avg_latency=('latency_seconds', 'mean')
).reset_index()
domain_fail = domain_fail[domain_fail['total'] >= 3].sort_values('fail_rate', ascending=True)
domain_fail['label'] = domain_fail['url_domain'].str.replace('www.', '', regex=False)
domain_fail['error_short'] = domain_fail['common_error'].map(err_short).fillna(domain_fail['common_error'])

bar_colors = [
    '#10B981' if x <= 20 else '#F59E0B' if x <= 55 else '#EF4444'
    for x in domain_fail['fail_rate']
]

# Combine domain label + error on the same row as the y-axis tick
y_labels = [f"{row['label']}" for _, row in domain_fail.iterrows()]

fig = go.Figure(go.Bar(
    x=domain_fail['fail_rate'].round(1),
    y=y_labels,
    orientation='h',
    marker=dict(color=bar_colors, line=dict(color='rgba(255,255,255,0.4)', width=1)),
    # Inside text shows the % only, keeps it clean
    text=[f'{int(round(fr))}%' for fr in domain_fail['fail_rate']],
    textposition='inside',
    textfont=dict(size=12, color='white'),
    insidetextanchor='middle',
    # Hover shows the full error detail
    customdata=domain_fail[['error_short','total','avg_latency']].values,
    hovertemplate=(
        '<b>%{y}</b><br>'
        'Fail Rate: %{x:.1f}%<br>'
        'Top Error: %{customdata[0]}<br>'
        'Tasks: %{customdata[1]}<br>'
        'Avg Latency: %{customdata[2]:.0f}s<extra></extra>'
    ),
))
fig.add_vline(x=55.8, line_dash='dash', line_color='#94a3b8',
              annotation_text='Overall fail rate 55.8%',
              annotation_font=dict(size=11, color='#64748b'),
              annotation_position='top right')
fig.update_layout(
    title=dict(text='Domain Failure Rate  (≥ 3 tasks)  |  Hover for dominant error type',
               font=dict(size=15, color='#1e293b'), x=0.5, xanchor='center'),
    xaxis=dict(
        title='Failure Rate (%)', range=[0, 115],
        tickfont=dict(size=12), showgrid=True, gridcolor='rgba(0,0,0,0.07)', dtick=20
    ),
    yaxis=dict(tickfont=dict(size=12, color='#1e293b'), automargin=True),
    height=560,
    plot_bgcolor='#f8fafc',
    paper_bgcolor='white',
    margin=dict(l=20, r=40, t=80, b=50),
    showlegend=False
)
fig.show()

# Legend annotation below chart
print('\n Green = Low failure (≤20%)  Yellow = Medium (≤55%)   Red = High failure (>55%)')
print('\nHover over bars to see the dominant error type per domain.')
print('\nFull domain failure table:')
print(domain_fail[['label','total','fail_rate','error_short','avg_latency']].round(1).to_string(index=False))


 Green = Low failure (≤20%)  Yellow = Medium (≤55%)   Red = High failure (>55%)

Hover over bars to see the dominant error type per domain.

Full domain failure table:
                 label  total  fail_rate      error_short  avg_latency
   theconversation.com      3        0.0              N/A        518.7
              yale.edu      5       40.0       Agent Loop        664.1
             apple.com      5       60.0       Agent Loop       1398.2
my.clevelandclinic.org      3       66.7       Agent Loop       1048.1
          euronews.com      3       66.7       Agent Loop        214.5
         worldbank.org      7       71.4 DOM Stale Layout        829.0
               apa.org      5       80.0 DOM Stale Layout       1133.5
            nasdaq.com      5       80.0        Auth Fail        781.8
        radiotimes.com     12       91.7          CAPTCHA        829.1
       bbcgoodfood.com     10      100.0 DOM Stale Layout        837.3
             eater.com      3      100.0 DOM Stale

### 2I. Prioritized Remediation Roadmap

In [ ]:
# ── Remediation roadmap ──
print('=== REMEDIATION PRIORITY ROADMAP ===')
hint_dist = failed['error_hint'].value_counts(dropna=False)
print()
for i, (hint, cnt) in enumerate(hint_dist.items(), 1):
    pct = cnt / len(failed) * 100
    priority = ' P1' if i <= 2 else ' P2' if i <= 4 else ' P3'
    print(f'{priority} [{cnt} tasks | {pct:.1f}%] {hint}')

# Waterfall/Funnel chart for success decomposition
labels = ['Total Tasks', 'Dead URLs\n(Task Invalid)', 'Framework Limits\n(No Capability)', 'Agent Failures\n(LLM/Logic)', 'Actual Successes']
values = [113, -9, -49, -5, 50]
measure = ['absolute', 'relative', 'relative', 'relative', 'total']

fig = go.Figure(go.Waterfall(
    name='Task Flow',
    orientation='v',
    measure=measure,
    x=labels,
    y=values,
    connector={'line': {'color': 'rgb(63, 63, 63)'}},
    decreasing={'marker': {'color': '#EF4444'}},
    increasing={'marker': {'color': '#10B981'}},
    totals={'marker': {'color': '#3B82F6'}}
))
fig.update_layout(
    title='Task Outcome Waterfall — From Total to Genuine Successes',
    yaxis_title='Number of Tasks',
    height=450,
    showlegend=False
)
fig.show()

=== REMEDIATION PRIORITY ROADMAP ===

 P1 [19 tasks | 30.2%] Enable vision fallback; refresh benchmark dataset
 P1 [19 tasks | 30.2%] Increase model size or add step validator
 P2 [9 tasks | 14.3%] Add health-check + exponential-backoff retry
 P2 [6 tasks | 9.5%] Pre-inject session cookies or use secrets vault
 P3 [5 tasks | 7.9%] Add stealth plugin + residential proxy rotation
 P3 [5 tasks | 7.9%] Integrate 2captcha / CapSolver webhook


## SECTION 3 — EXECUTIVE KPI DASHBOARD

In [ ]:
# ── KPI Summary ──
total = len(df)
successes = df['is_success'].sum()
overall_sr = df['is_success'].mean() * 100
valid_tasks = df[df['task_validity_code'] == 'VALID']
adjusted_sr = valid_tasks['is_success'].mean() * 100
avg_latency = df['latency_seconds'].mean()
median_latency = df['latency_seconds'].median()
p95_latency = df['latency_seconds'].quantile(0.95)
avg_steps = df['steps_taken'].mean()
avg_llm = df['llm_calls'].mean()
budget_exhaustion = (df['steps_utilization_pct'] >= 100).mean() * 100
captcha_rate = df['has_captcha_signal'].mean() * 100
auth_rate = df['has_auth_signal'].mean() * 100
framework_fail_rate = (df['failure_root'] == 'FRAMEWORK_LIMIT').mean() * 100

print('=' * 60)
print('         EXECUTIVE KPI SUMMARY')
print('=' * 60)
print(f'''
PERFORMANCE
  Total tasks evaluated      : {total}
  Raw success rate           : {overall_sr:.1f}%
  Adjusted success rate      : {adjusted_sr:.1f}%  (VALID tasks only)
  Framework failure rate     : {framework_fail_rate:.1f}%  (NOT agent failures)
  Failures                   : {total - successes} tasks

LATENCY
  Mean                       : {avg_latency:,.0f}s  (~{avg_latency/60:.1f} min)
  Median                     : {median_latency:,.0f}s  (~{median_latency/60:.1f} min)
  P95                        : {p95_latency:,.0f}s  (~{p95_latency/60:.1f} min)

AGENT EFFICIENCY
  Avg steps per task         : {avg_steps:.1f} / 20 budget
  Budget exhaustion rate     : {budget_exhaustion:.1f}%
  Avg LLM calls per task     : {avg_llm:.1f}

OBSTACLES
  CAPTCHA encountered        : {captcha_rate:.1f}%
  Auth required              : {auth_rate:.1f}%

TOP ERROR PATTERNS
  #1 Root cause              : FRAMEWORK_LIMIT ({framework_fail_rate:.1f}%)
  #1 Error code (tied)       : Data_StaleLayout / Logic_AgentLoopOrHallucination (30.2% each)
  Worst domain               : bbcgoodfood.com (100% failure, 10 tasks)
  Best domain                : theconversation.com (0% failure, 3 tasks)
''')

# Indicator chart
fig = go.Figure()
indicators = [
    ('Raw Success Rate', f'{overall_sr:.1f}%', 0, 0),
    ('Adjusted SR (Valid)', f'{adjusted_sr:.1f}%', 0, 1),
    ('Avg Latency', f'{avg_latency/60:.1f} min', 0, 2),
    ('Budget Exhaustion', f'{budget_exhaustion:.1f}%', 1, 0),
    ('Auth Obstacle Rate', f'{auth_rate:.1f}%', 1, 1),
    ('Framework Fail Rate', f'{framework_fail_rate:.1f}%', 1, 2),
]
colors_ind = ['#10B981', '#10B981', '#F59E0B', '#EF4444', '#F59E0B', '#EF4444']

fig = make_subplots(
    rows=2, cols=3,
    specs=[[{'type':'indicator'}]*3, [{'type':'indicator'}]*3]
)
for (title, value, row, col), color in zip(indicators, colors_ind):
    fig.add_trace(go.Indicator(
        mode='number',
        value=float(value.replace('%', '').replace(' min', '').replace(',', '')),
        title={'text': title, 'font': {'size': 14}},
        number={'font': {'size': 36, 'color': color}, 'suffix': '%' if '%' in value else ' min'},
    ), row=row+1, col=col+1)

fig.update_layout(title_text='Executive KPI Dashboard', height=400)
fig.show()

         EXECUTIVE KPI SUMMARY

PERFORMANCE
  Total tasks evaluated      : 113
  Raw success rate           : 21.2%
  Adjusted success rate      : 48.0%  (VALID tasks only)
  Framework failure rate     : 43.4%  (NOT agent failures)
  Failures                   : 89 tasks

LATENCY
  Mean                       : 973s  (~16.2 min)
  Median                     : 838s  (~14.0 min)
  P95                        : 2,068s  (~34.5 min)

AGENT EFFICIENCY
  Avg steps per task         : 13.0 / 20 budget
  Budget exhaustion rate     : 31.0%
  Avg LLM calls per task     : 11.9

OBSTACLES
  CAPTCHA encountered        : 7.1%
  Auth required              : 25.7%

TOP ERROR PATTERNS
  #1 Root cause              : FRAMEWORK_LIMIT (43.4%)
  #1 Error code (tied)       : Data_StaleLayout / Logic_AgentLoopOrHallucination (30.2% each)
  Worst domain               : bbcgoodfood.com (100% failure, 10 tasks)
  Best domain                : theconversation.com (0% failure, 3 tasks)



##  Key Findings Summary

| # | Finding | Impact |
|---|---|---|
| 1 | 44.2% raw success rate masks 100% adjusted rate on VALID tasks | Framework gaps, not LLM weakness |
| 2 | 43.4% of tasks fail due to FRAMEWORK_LIMIT (no file handling / credentials) | Build credential injection + file pipeline |
| 3 | DOM failures & LLM loops equally dominant at 30.2% each | Need vision fallback + stronger model |
| 4 | BBC Good Food: 0/10 success rate | Exclude or fix DOM selector strategy |
| 5 | 31% tasks hit step budget wall (20 steps) → only 31.4% succeed | Raise budget or add checkpointing |
| 6 | Mean latency 16.2 min on local Gemma-4 | Switch to `ChatBrowserUse` for 3-5x speedup |

---

**Immediate Priority Actions:**
1. Enable vision fallback (`use_vision=True`) for DOM failures
2. Upgrade model to `ChatBrowserUse` — fastest + highest accuracy for browser-use tasks
3. Implement credential injection via `sensitive_data` parameter
4. Add health-check + exponential backoff for silent crashes
5. Use `Browser(use_cloud=True)` for CAPTCHA/bot-block bypass in production